# 02 — Model comparison (2 baselines + 5 deep)

Compares every model trained against the unified 2019-2026 cache
(`cache/goes_features_2019_2026`, temporal split train 2019-2024 / val 2025 / test 2026):

- **baselines** (location-free per-cell features): logistic regression, XGBoost
- **deep** (image + per-cell features, two-branch): R(2+1)D, small 3D-ResNet, 3D-CNN,
  CNN+temporal-attention, ConvLSTM

Run **after** training the models. Each trainer writes its artifact to `outputs/`
(`<name>.pt` deep, `<name>.pkl` tabular) + `<name>_results.npz` (deep epoch curves).
This notebook reloads each one, runs inference on val + test, and reports AUPRC and
P/R/F1/CSI (exact + 1-grid neighbourhood), training curves, and prediction maps. The
**base rate** (positive fraction) is the reference floor.

To train first (from `notebooks/model/`):

```
NCCL_P2P_DISABLE=1 torchrun --nproc_per_node=2 trainers/cnn3d.py     # + r2plus1d, resnet3d, cnn_attn, convlstm
python trainers/logreg.py    # tabular: plain python, NOT torchrun
python trainers/xgb.py
```

## 0. Setup, registry, and artifact availability

In [ ]:
import pickle
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
OUT_DIR = MODEL_DIR / "outputs"
sys.path.insert(0, str(MODEL_DIR))
sys.path.insert(0, str(MODEL_DIR / "trainers"))

from config import STATES_GEOJSON, build_grid_cells
from gridindex import build_pix2cell
import cnn3d, r2plus1d, resnet3d, cnn_attn, convlstm   # noqa: E401  (deep trainers)
import logreg, xgb                                      # noqa: E401  (tabular trainers)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# shared grid / land mask / splits (the same filtered split every trainer uses)
p2c, GRID_R, GRID_C, land = build_pix2cell()
tr_days, va_days, te_days = cnn3d.load_splits()
CACHE_DIR = cnn3d.CACHE_DIR


def base_rate(days):
    y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days])
    return float(y[:, land].mean())


BASE = {"val": base_rate(va_days), "test": base_rate(te_days)}

# grid polygons + state outlines (Albers) for the maps
cells_gdf, _, _, _ = build_grid_cells()
cells_albers = cells_gdf.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)

# 7-model registry: deep (reload .pt into FloodNet) + tabular (reload .pkl)
MODELS = {
    "R(2+1)D":   dict(kind="deep", mod=r2plus1d, ckpt="r2plus1d.pt",
                      res="r2plus1d_results.npz", c="#1d6fb8", cmap="Blues"),
    "3D-ResNet": dict(kind="deep", mod=resnet3d, ckpt="resnet3d.pt",
                      res="resnet3d_results.npz", c="#d62828", cmap="Reds"),
    "3D-CNN":    dict(kind="deep", mod=cnn3d, ckpt="cnn3d.pt",
                      res="cnn3d_results.npz", c="#e09f3e", cmap="Oranges"),
    "CNN+attn":  dict(kind="deep", mod=cnn_attn, ckpt="cnn_attn.pt",
                      res="cnn_attn_results.npz", c="#2a9d8f", cmap="GnBu"),
    "ConvLSTM":  dict(kind="deep", mod=convlstm, ckpt="convlstm.pt",
                      res="convlstm_results.npz", c="#6a4c93", cmap="Purples"),
    "LogReg":    dict(kind="tab", mod=logreg, pkl="logreg.pkl",
                      c="#8d99ae", cmap="bone_r"),
    "XGBoost":   dict(kind="tab", mod=xgb, pkl="xgb.pkl",
                      c="#386641", cmap="YlGn"),
}


def artifact(mi):
    return OUT_DIR / (mi["ckpt"] if mi["kind"] == "deep" else mi["pkl"])


AVAIL = [n for n, mi in MODELS.items() if artifact(mi).exists()]

print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells")
print(f"days: train {len(tr_days)}  val {len(va_days)}  test {len(te_days)}")
print(f"base rate: val {BASE['val']:.4f}  test {BASE['test']:.4f}")
for n, mi in MODELS.items():
    print(f"  {n:10s} {mi['kind']:4s} -> {'TRAINED' if n in AVAIL else 'missing (train it)'}")

## 1. Training curves (deep models)

Loss and PR-AUC over the 2 epochs, train/val/test, from each `*_results.npz`. The dotted
line is the test base rate. (Tabular baselines have no epoch curves.)

In [ ]:
SPLITS = [("train", "#1d6fb8", "-o"), ("val", "#e09f3e", "-s"), ("test", "#2a9d8f", "-^")]
deep_avail = [n for n in AVAIL if MODELS[n]["kind"] == "deep"]

for name in deep_avail:
    path = OUT_DIR / MODELS[name]["res"]
    if not path.exists():
        print(f"{name}: no results npz"); continue
    h = np.load(path)["hist"]          # epoch, lr, tr/va/te loss, tr/va/te PR-AUC
    if h.ndim != 2 or h.shape[1] != 8 or len(h) == 0:
        print(f"{name}: empty/bad hist"); continue
    ep = h[:, 0]
    fig, (a_loss, a_pr) = plt.subplots(1, 2, figsize=(13, 4.4))
    for j, (split, col, mk) in enumerate(SPLITS):
        a_loss.plot(ep, h[:, 2 + j], mk, color=col, ms=4, label=split)
        a_pr.plot(ep, h[:, 5 + j], mk, color=col, ms=4, label=split)
    a_pr.axhline(BASE["test"], ls=":", color="black", lw=1.4,
                 label=f"base rate ({BASE['test']:.4f})")
    a_loss.set(xlabel="epoch", ylabel="combined loss", title=f"{name} - loss")
    a_pr.set(xlabel="epoch", ylabel="PR-AUC (land)", title=f"{name} - PR-AUC")
    for ax in (a_loss, a_pr):
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    b = int(h[:, 6].argmax())
    print(f"{name}: best val PR-AUC {h[b,6]:.4f} @ epoch {int(h[b,0])} (test {h[b,7]:.4f})")

## 2. Inference on val + test

Reload each trained model and predict per-cell probabilities `(N, 59, 95)`. Deep models
reload their `.pt` into `FloodNet`; tabular models reload their `.pkl` and use
`predict_grids`. Cached once for the metrics table and maps below.

In [ ]:
@torch.no_grad()
def infer(name, days):
    """Reload model `name` and predict -> (probs, trues), each (len(days), 59, 95)."""
    mi = MODELS[name]
    if mi["kind"] == "deep":
        mod = mi["mod"]
        sub = mod._subsample_index(p2c, mod.ENC_H, mod.ENC_W)
        net = mod.FloodNet(sub, GRID_R, GRID_C).to(DEVICE)
        net.load_state_dict(torch.load(OUT_DIR / mi["ckpt"], map_location=DEVICE))
        net.eval()
        probs, trues = [], []
        loader = DataLoader(mod.FeatureCache(days), batch_size=1, num_workers=8)
        for img, goes, glm, daily, t, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                p = torch.sigmoid(net(img.to(DEVICE).float(), goes.to(DEVICE).float(),
                                      glm.to(DEVICE).float(), daily.to(DEVICE).float(),
                                      t.to(DEVICE).float())).squeeze(1)
            probs.append(p.float().cpu().numpy()[0]); trues.append(y.numpy()[0])
        del net; torch.cuda.empty_cache()
        return np.array(probs), np.array(trues)
    # tabular
    payload = pickle.load(open(OUT_DIR / mi["pkl"], "rb"))
    return mi["mod"].predict_grids(payload, days, land)


val_pred, test_pred = {}, {}
for name in AVAIL:
    print(f"inferring {name} ...", end=" ", flush=True)
    val_pred[name] = infer(name, va_days)
    test_pred[name] = infer(name, te_days)
    print("done")

## 3. Metrics table

AUPRC + P/R/F1/CSI (exact and 1-grid neighbourhood) on val and test. The threshold is the
best-F1 point on **val**, applied to test (the honest operating point). `xbase` = AUPRC /
base rate. 1-grid metrics credit a prediction within one cell of a true flood.

In [ ]:
import pandas as pd


def _dilate_1grid(mask):
    out = mask.copy()
    out[:-1, :] |= mask[1:, :]
    out[1:, :] |= mask[:-1, :]
    out[:, :-1] |= mask[:, 1:]
    out[:, 1:] |= mask[:, :-1]
    out[:-1, :-1] |= mask[1:, 1:]
    out[1:, 1:] |= mask[:-1, :-1]
    out[:-1, 1:] |= mask[1:, :-1]
    out[1:, :-1] |= mask[:-1, 1:]
    return out


def full_metrics(probs, trues, threshold=None):
    p = probs[:, land].ravel()
    t = trues[:, land].ravel().astype(np.int32)
    prauc = average_precision_score(t, p)
    if threshold is None:
        pr_c, rc_c, thr_c = precision_recall_curve(t, p)
        f1_c = 2 * pr_c * rc_c / (pr_c + rc_c + 1e-9)
        threshold = float(thr_c[np.argmax(f1_c[:-1])])
    yb = (p >= threshold).astype(np.int32)
    tp = int((yb * t).sum()); fp = int((yb * (1 - t)).sum()); fn = int(((1 - yb) * t).sum())
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9); csi = tp / (tp + fn + fp + 1e-9)
    h_nb = m_nb = fa_nb = 0
    for i in range(len(probs)):
        yt = (trues[i] > 0.5) & land
        yp = (probs[i] >= threshold) & land
        h_nb += int((yt & (_dilate_1grid(yp) & land)).sum())
        m_nb += int((yt & ~(_dilate_1grid(yp) & land)).sum())
        fa_nb += int((yp & ~(_dilate_1grid(yt) & land)).sum())
    prec1 = h_nb / (h_nb + fa_nb + 1e-9); rec1 = h_nb / (h_nb + m_nb + 1e-9)
    f1_1 = 2 * prec1 * rec1 / (prec1 + rec1 + 1e-9); csi1 = h_nb / (h_nb + m_nb + fa_nb + 1e-9)
    return dict(prauc=prauc, thr=threshold, prec=prec, rec=rec, f1=f1, csi=csi,
                prec1=prec1, rec1=rec1, f1_1=f1_1, csi1=csi1)


rows = []
THR = {}
for name in AVAIL:
    vm = full_metrics(*val_pred[name], threshold=None)
    THR[name] = vm["thr"]
    tm = full_metrics(*test_pred[name], threshold=vm["thr"])
    for split, m in [("val", vm), ("test", tm)]:
        rows.append(dict(model=name, split=split, AUPRC=m["prauc"],
                         xbase=m["prauc"] / BASE[split], thr=m["thr"],
                         P=m["prec"], R=m["rec"], F1=m["f1"], CSI=m["csi"],
                         P1=m["prec1"], R1=m["rec1"], F1_1=m["f1_1"], CSI1=m["csi1"]))

tbl = pd.DataFrame(rows)
test_tbl = tbl[tbl.split == "test"].sort_values("AUPRC", ascending=False)
print(f"base rate: val {BASE['val']:.4f}  test {BASE['test']:.4f}\n")
print("TEST (sorted by AUPRC):")
fmt = {c: "{:.3f}".format for c in ["AUPRC", "xbase", "thr", "P", "R", "F1", "CSI",
                                    "P1", "R1", "F1_1", "CSI1"]}
display(test_tbl.set_index("model").drop(columns="split").style.format(fmt))
tbl.round(4)

## 4. Prediction maps — random test days

Ground truth vs each model's binary prediction (at its best-val-F1 threshold), plus a
hit/miss/false-alarm error map for the best deep model. Per-day AUPRC under each panel.

In [ ]:
assert AVAIL, "no trained models yet - train them first (see section 0)"
N_SHOW = 3
SEED = 7

ERR_CMAP = mcolors.ListedColormap(["#e8e8e8", "#2a9d8f", "#e63946", "#f4a261"])
ERR_NORM = mcolors.BoundaryNorm([0, 0.5, 1.5, 2.5, 3.5], 4)


def _draw(ax, values, cmap, norm=None, title=None, ylabel=None, sub=None):
    gdf = cells_albers.copy(); gdf["v"] = values[RR, CC]
    gdf.boundary.plot(ax=ax, color="white", lw=0.1, zorder=2)
    kw = dict(norm=norm) if norm is not None else dict(vmin=0, vmax=1)
    gdf.plot(column="v", cmap=cmap, ax=ax, zorder=1, edgecolor="none", **kw)
    states.boundary.plot(ax=ax, color="0.5", lw=0.5, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:  ax.set_title(title, fontsize=10, fontweight="bold")
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)
    if sub:    ax.set_xlabel(sub, fontsize=8, color="0.3")


# best deep model by test AUPRC (for the error map)
deep_avail = [n for n in AVAIL if MODELS[n]["kind"] == "deep"]
best_deep = (max(deep_avail, key=lambda n: full_metrics(*test_pred[n], threshold=THR[n])["prauc"])
             if deep_avail else None)

rng = np.random.default_rng(SEED)
sel = sorted(rng.choice(len(te_days), size=min(N_SHOW, len(te_days)), replace=False))
ncol = 1 + len(AVAIL) + (1 if best_deep else 0)
fig, axes = plt.subplots(len(sel), ncol, figsize=(2.6 * ncol, 2.5 * len(sel)))
if len(sel) == 1:
    axes = axes[None, :]

for r, i in enumerate(sel):
    gt = test_pred[AVAIL[0]][1][i]
    yt = gt[land].ravel().astype(int)
    _draw(axes[r, 0], gt, "Greens", title="Ground truth" if r == 0 else None,
          ylabel=te_days[i], sub=f"{int((gt[land] > 0).sum())} flood cells")
    for j, name in enumerate(AVAIL, start=1):
        pr = test_pred[name][0][i]
        ap = average_precision_score(yt, pr[land].ravel()) if yt.sum() else float("nan")
        _draw(axes[r, j], (pr >= THR[name]).astype(float), MODELS[name]["cmap"],
              title=name if r == 0 else None, sub=f"AUPRC {ap:.3f}")
    if best_deep:
        pr = test_pred[best_deep][0][i]
        yt2 = (gt > 0.5) & land; yp2 = (pr >= THR[best_deep]) & land
        err = np.zeros((GRID_R, GRID_C)); err[yt2 & yp2] = 1; err[yt2 & ~yp2] = 2
        err[~yt2 & yp2] = 3
        _draw(axes[r, -1], err, ERR_CMAP, norm=ERR_NORM,
              title=f"{best_deep} hit/miss/FA" if r == 0 else None)

fig.suptitle("Test days - ground truth | model predictions | error "
             "(green=hit, red=miss, orange=false alarm)", fontsize=12, y=1.01)
plt.tight_layout(); plt.show()